# The book nobody has read yet

Unit 8 gave us the book's best recommender so far: a **two-tower** neural model -- a reader-ID
embedding tower and a book-ID embedding tower, trained in PyTorch with BPR, landing at hit@10
**~0.340** on `val`. It wins because it learns a taste vector for every reader and every book *from
the interaction log*.

And that is exactly where it fails. A tower that learns a book's vector *from interactions* has
nothing to learn from for a book that has **no interactions** -- a brand-new title, a just-added
back-catalog book, anything the training log never saw a single read of. In our slice, **818 of the
2,000 catalog books have zero train-positive interactions** (41% of the shelf). The two-tower does not
*skip* them -- it embeds every book -- but a book that only ever appeared while being *sampled as a
negative* gets an embedding pushed toward the bottom. The model learns to **bury** it.

That is the **item cold-start** problem, flagged all the way back in Unit 6's coverage discussion and
never solved. This unit solves it. The idea: stop describing a book only by *who read it* and start
describing it by *what it is* -- its genre, its author, its keywords -- so a book with no reads still
gets a sensible vector. We will build that **feature tower**, find the one design knob that makes or
breaks it (the **negative pool**), and measure -- honestly -- what it costs. The headline is not a new
warm record. It is a **trade**, and the trade is the lesson.

## Load the substrate and name the cold items

The Units 1-8 substrate is unchanged: `generated_dir()` locates the seeded slice, the catalog, the
gzip'd interaction log, the keyword corpus and the committed GloVe subset load as before, and
`cold_partitions.json` lists the cold readers we always exclude from scoring. (`test` stays sealed for
Checkpoint B; we only ever score on `val`.)

The one new quantity is the **cold-item set**: every catalog book with **zero training positives**.
That set is the denominator for everything in this unit, and -- crucially -- it is defined without
touching `val`, so no metric we build can secretly leak the validation split.

In [1]:
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from bookrec import (
    FeatureTowerRetrievalPath,
    ItemItemRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    generated_dir,
    load_catalog,
    load_glove_subset,
    load_keywords,
    run_validation_scoreboard,
)

data_dir = generated_dir()
catalog = load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")
keywords = load_keywords(data_dir / "keywords.csv.gz")
glove = load_glove_subset()
cold_readers = set(json.loads((data_dir / "cold_partitions.json").read_text())["cold_readers"])

# Per-reader train history (the "seen" set) and val positives -- both from positive rows only.
seen_by = defaultdict(set)
val_by = defaultdict(set)
train_positive_items = set()
for row in rows:
    if int(row["label"]) != 1:
        continue
    reader_id, item_id = int(row["reader_id"]), int(row["item_id"])
    if row["split"] == "train":
        seen_by[reader_id].add(item_id)
        train_positive_items.add(item_id)
    elif row["split"] == "val":
        val_by[reader_id].add(item_id)

# The cold-item set: every catalog book with ZERO training positives (defined without val).
zero_train = set(catalog_ids) - train_positive_items
print(f"catalog {len(catalog_ids)} books; {len(rows)} interaction rows; {len(cold_readers)} cold readers")
print(f"cold items (zero training positives): {len(zero_train)} "
      f"({100 * len(zero_train) / len(catalog_ids):.0f}% of the catalog)")

catalog 2000 books; 27004 interaction rows; 60 cold readers
cold items (zero training positives): 818 (41% of the catalog)


## 1. Make the cold-item problem concrete

First, let us *watch* the two-tower bury the cold items, so the fix has something to beat.

We need a metric that works when the cold items have **no val positives** -- and by construction they
do not (the generator never exposes a designated cold item outside the sealed `test` split, so there
is nothing to score a hit against on `val`). Measuring *relevance* on cold items is therefore
impossible here; it waits for Checkpoint B on `test`. What we *can* measure, with no val dependency
at all, is **coverage**:

> **cold-item coverage** = the number of distinct cold items that appear in *some* reader's top-10,
> divided by 818 -- counted over **every fitted non-cold reader, scored from that reader's train
> history only**.

Coverage asks the honest question: does the recommender ever surface a cold book at all? Fit Unit 8's
ID-only two-tower and Unit 4's item-item CF -- the two *collaborative* paths -- and measure it.

In [2]:
def fitted_cohort(path):
    # All fitted non-cold readers: the ones the path learned an embedding for, minus cold readers.
    try:  # the towers expose their fitted reader ids in the artifact
        fitted = [int(r) for r in path.artifact()["reader_ids"]]
    except KeyError:  # item-item CF has no reader embedding -> use the train cohort directly
        fitted = sorted(seen_by)
    return [r for r in fitted if r not in cold_readers]


def cold_coverage(path):
    # Distinct cold items surfaced in any fitted reader's top-10 / 818 (train-only scoring).
    surfaced = set()
    for reader_id in fitted_cohort(path):
        for candidate in path.retrieve(reader_id, {"seen": seen_by[reader_id]}, 10):
            if candidate.item_id in zero_train:
                surfaced.add(candidate.item_id)
    return len(surfaced) / len(zero_train), len(surfaced)


id_two_tower = TwoTowerRetrievalPath(seed=0).fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)

for name, path in (("ID-only two-tower (U8)", id_two_tower), ("item-item CF (U4)", item_item)):
    coverage, surfaced = cold_coverage(path)
    print(f"{name:>24}: cold coverage {coverage:.4f} ({surfaced} of {len(zero_train)} cold items)")
print()
print("Both collaborative paths surface EXACTLY ZERO cold items. A book with no reads has an id")
print("embedding that was only ever shaped by being sampled as a negative -- so it ranks at the floor.")

  ID-only two-tower (U8): cold coverage 0.0000 (0 of 818 cold items)


       item-item CF (U4): cold coverage 0.0000 (0 of 818 cold items)

Both collaborative paths surface EXACTLY ZERO cold items. A book with no reads has an id
embedding that was only ever shaped by being sampled as a negative -- so it ranks at the floor.


## 2. The feature tower: describe a book by what it *is*

The id embedding is a dead end for a cold book -- it has no train-positive interactions to learn a
taste direction from (only negative samples, which push it down), and no amount of extra training can
conjure one. So we give the item tower a second source of signal
that *every* book has, read or not: its **content features**.

The two-tower model does not change shape. The reader tower stays a bare id embedding, and the score
is still `reader_emb . item_emb`. What changes is how the **item** vector is built. Instead of a lone
id row, the item tower **sums four pieces**, all projected to the shared dimension:

```
item_emb = id_embedding(book)          # the Unit-8 collaborative row (useless for a cold book)
         + genre_embedding(book)       # learned, mean-pooled over the book's genres
         + author_embedding(book)      # learned, one row per author
         + Linear(glove_keyword_vector(book))   # the Unit-7 GloVe vector, projected to the dim
```

The genre, author and GloVe-projection parameters are **shared across all books** and trained on the
**warm** items -- so a cold book, whose id row only ever receives negative gradients, still inherits a sensible vector
from its genre, author and keywords. That is the whole idea. Build the composition once on one real
cold book (book 139) to see the four pieces line up and sum -- the embeddings are random here;
training is what will make the sum meaningful.

In [3]:
import torch

from bookrec.lexical import tokenize
from bookrec import book_embedding

torch.manual_seed(0)
dim = 32

# Genre and author vocabularies, read off the catalog itself (sorted, so they are deterministic).
genre_vocab = sorted({g.strip() for book in catalog.values()
                      for g in book.fields.get("genres", "").split(";") if g.strip()})
author_vocab = sorted({book.fields.get("author_id", "") for book in catalog.values()})
genre_index = {g: j for j, g in enumerate(genre_vocab)}
author_index = {a: j for j, a in enumerate(author_vocab)}
print(f"{len(genre_vocab)} genres, {len(author_vocab)} authors, GloVe dim {glove.dim}")

# The four learned building blocks (random init -- this is the architecture, not a trained model).
id_tower = torch.nn.Embedding(len(catalog_ids), dim)
genre_tower = torch.nn.Embedding(len(genre_vocab), dim)
author_tower = torch.nn.Embedding(len(author_vocab), dim)
glove_proj = torch.nn.Linear(glove.dim, dim)

# Assemble ONE book's item vector -- book 139, a cold "children" title with zero train-positive reads (it appears only as sampled negatives).
book = catalog[139]
genre_multihot = np.zeros(len(genre_vocab), dtype=np.float32)
for g in book.fields.get("genres", "").split(";"):
    if g.strip():
        genre_multihot[genre_index[g.strip()]] = 1.0
author_idx = author_index[book.fields.get("author_id", "")]
glove_vector = book_embedding(tokenize(keywords.get(139, "")), glove).astype(np.float32)

# ids are 0..1999 here, so id == row; the shipped path maps ids->rows explicitly via row_of.
id_vec = id_tower(torch.from_numpy(np.array([139], dtype=np.int64)))[0]
genre_vec = torch.from_numpy(genre_multihot) @ genre_tower.weight / max(1.0, genre_multihot.sum())
author_vec = author_tower(torch.from_numpy(np.array([author_idx], dtype=np.int64)))[0]
glove_vec = glove_proj(torch.from_numpy(glove_vector))
item_vec = id_vec + genre_vec + author_vec + glove_vec

print(f"book 139 '{book.title}' | genres {book.fields.get('genres')} | author {book.fields.get('author_id')}")
print(f"id {tuple(id_vec.shape)} + genre {tuple(genre_vec.shape)} + author {tuple(author_vec.shape)} "
      f"+ Linear(glove) {tuple(glove_vec.shape)} = item {tuple(item_vec.shape)}")
print("Four pieces, each a 32-vector, summed into one item embedding -- the cold book now has a vector.")

12 genres, 300 authors, GloVe dim 100
book 139 'Book 00139' | genres children | author 173
id (32,) + genre (32,) + author (32,) + Linear(glove) (32,) = item (32,)
Four pieces, each a 32-vector, summed into one item embedding -- the cold book now has a vector.


## 3. The negative pool: the one knob that makes it work

Here is the beat the whole unit turns on. The feature tower *can* give a cold book a vector -- but
whether that vector is any good depends on a design choice hidden inside BPR training: **where do the
sampled negatives come from?**

Unit 8 sampled negatives **uniformly over the full catalog**. That was harmless when every book was
just an id. But 41% of our catalog is cold, so under full-catalog sampling a cold book is *only ever*
drawn as a **negative** -- never a positive. Its feature parameters get mostly a "push this down"
gradient, and the tower suppresses it much as the id-only tower did. This is not *categorical* -- under
full-catalog sampling the feature tower still surfaces 39 of the 818 cold books -- but it is strong:
features can only weakly counter an item's own negative gradient.

The fix is to draw negatives **only from the warm (train-positive) item universe**, so a cold book is
never sampled as a negative and its features are free to place it. This is the shipped default,
`negative_pool="warm"`. Train the feature tower both ways and compare -- watch the cold coverage, and
watch the honest cost on warm hit@10.

In [4]:
feature_tower = FeatureTowerRetrievalPath(  # the shipped default: negatives from the WARM universe
    catalog, keywords, glove, n_epochs=40, negative_pool="warm", seed=0
).fit(rows, catalog=catalog_ids)

feature_full = FeatureTowerRetrievalPath(  # Unit 8's sampler / the failure mode: full-catalog negs
    catalog, keywords, glove, n_epochs=40, negative_pool="full", seed=0
).fit(rows, catalog=catalog_ids)


def warm_hit(path):
    return run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
    ).hit_rate_at_k


warm_cov, warm_n = cold_coverage(feature_tower)
full_cov, full_n = cold_coverage(feature_full)
print(f"{'negative pool':>14} {'warm hit@10':>12} {'cold coverage':>14}")
print(f"{'warm-only':>14} {warm_hit(feature_tower):>12.4f} {warm_cov:>11.4f} ({warm_n})")
print(f"{'full-catalog':>14} {warm_hit(feature_full):>12.4f} {full_cov:>11.4f} ({full_n})")
print()
print("Full-catalog negatives bury cold items (coverage ~0.05) while scoring warm ~0.34, same as U8.")
print("Warm-only negatives lift cold coverage ~3x (to ~0.14) at a small warm cost (~0.30). A TRADE.")

 negative pool  warm hit@10  cold coverage


     warm-only       0.2980      0.1381 (113)


  full-catalog       0.3380      0.0477 (39)

Full-catalog negatives bury cold items (coverage ~0.05) while scoring warm ~0.34, same as U8.
Warm-only negatives lift cold coverage ~3x (to ~0.14) at a small warm cost (~0.30). A TRADE.


## 4. Hard negatives: a sharper knob, not a free lunch

A natural next thought: if sampling matters this much, why sample negatives *uniformly* at all? A
**hard negative** is an informative one -- a book the model might wrongly rank high -- so that each
BPR step corrects a real mistake. A cheap proxy for "hard" is **popularity**: weight the negative
sampler toward frequently-read books (`negative_pool="hard"`, still drawn over the warm universe).

Two cautions come with it. First, popularity-weighting can draw a book the reader actually *would*
like but simply has not read yet -- a **false negative** -- and train the model to push a true
positive down. Second, "harder" is not automatically "better". Measure it here, and read the result
honestly: on this small log, hard negatives trade warm accuracy for cold reach *even harder* than
warm-only did -- they do **not** sharpen both at once. It is an ablation knob, not an upgrade.

In [5]:
feature_hard = FeatureTowerRetrievalPath(  # popularity-weighted negatives over the warm universe
    catalog, keywords, glove, n_epochs=40, negative_pool="hard", seed=0
).fit(rows, catalog=catalog_ids)

hard_cov, hard_n = cold_coverage(feature_hard)
print(f"{'negative pool':>14} {'warm hit@10':>12} {'cold coverage':>14}")
print(f"{'warm-only':>14} {warm_hit(feature_tower):>12.4f} {warm_cov:>11.4f} ({warm_n})")
print(f"{'hard (popularity)':>14} {warm_hit(feature_hard):>12.4f} {hard_cov:>11.4f} ({hard_n})")
print()
print("Hard negatives push cold coverage higher still (~0.23) but drop warm hit further (~0.17):")
print("the trade gets steeper, not free. Default stays warm-only; hard is the ablation/caveat knob.")

 negative pool  warm hit@10  cold coverage


     warm-only       0.2980      0.1381 (113)


hard (popularity)       0.1720      0.2262 (185)

Hard negatives push cold coverage higher still (~0.23) but drop warm hit further (~0.17):
the trade gets steeper, not free. Default stays warm-only; hard is the ablation/caveat knob.


## Reveal: `FeatureTowerRetrievalPath`, scored honestly

The warm-only path we trained above *is* the shipped `FeatureTowerRetrievalPath`. Like every other
`RetrievalPath` it takes its content features through the constructor (the catalog books, keywords and
GloVe subset, exactly like Unit 7's semantic path) and keeps the plain `fit` / `retrieve` protocol;
torch is imported only inside `fit`, and `retrieve` scores from a composed numpy item matrix, so it
stays torch-free.

Now the real question, answered the way every unit answers it: run the frozen `val` scoreboard over
the field -- item-item CF, Unit 7's semantic content path, Unit 8's ID-only two-tower, and our feature
tower -- at **warm hit@10**, and beside it the **cold-item coverage** of each.

In [6]:
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)

board = {
    "item-item CF (U4)": item_item,
    "semantic (U7)": semantic,
    "ID-only two-tower (U8)": id_two_tower,
    "feature tower (U9)": feature_tower,
}
print(f"{'path':>24} {'warm hit@10':>12} {'cold coverage':>16}")
for name, path in board.items():
    coverage, surfaced = cold_coverage(path)
    print(f"{name:>24} {warm_hit(path):>12.4f} {coverage:>11.4f} ({surfaced:>3})")

                    path  warm hit@10    cold coverage


       item-item CF (U4)       0.2520      0.0000 (  0)


           semantic (U7)       0.1020      0.3276 (268)


  ID-only two-tower (U8)       0.3400      0.0000 (  0)


      feature tower (U9)       0.2980      0.1381 (113)


Read the two columns together -- that is the whole story of this unit.

- **Warm hit@10**: the feature tower (**~0.298**) sits just below the ID-only two-tower (**~0.340**) --
  it gives up about four points of warm accuracy. It did **not** set a new record, and it was never
  going to: the id row still does the warm work, and we have diluted it with features.
- **Cold coverage**: the two *collaborative* paths -- CF and the ID-only two-tower -- surface
  **exactly zero** cold items. The feature tower surfaces **~0.138** (113 of 818). That is the win,
  and it is a win no id-only collaborative model can have.

And notice the honest comparison the semantic row forces: Unit 7's content path already covers cold
items *freely* (**~0.33**), because it never looked at interactions in the first place -- it just
reads far worse on warm hit (**~0.10**). So the feature tower is **not** "the first path to serve cold
items". It is the first **learned-taste / collaborative** path to serve cold items *while staying near
the book's best warm hit* -- a deliberate **compromise**, not dominance. (As always: one read, on
`val`, at this pinned config, with `test` sealed.)

### One cold book, surfaced for one reader

Coverage is an aggregate; make it concrete. Walk the readers who happen to have a cold book as their
held-out `val` read (the incidental cold-val cohort -- report-only, small-n, never a gate) and find
the first one the feature tower actually serves that cold next-read to. Then check the ID-only
two-tower and CF on the same reader.

In [7]:
for reader_id in sorted(val_by):
    if reader_id in cold_readers:
        continue
    seen = seen_by[reader_id]
    cold_relevant = (val_by[reader_id] - seen) & zero_train
    if not cold_relevant:
        continue
    feature_recs = {c.item_id for c in feature_tower.retrieve(reader_id, {"seen": seen}, 10)}
    surfaced = feature_recs & cold_relevant
    if surfaced:
        book_id = sorted(surfaced)[0]
        tower_recs = {c.item_id for c in id_two_tower.retrieve(reader_id, {"seen": seen}, 10)}
        cf_recs = {c.item_id for c in item_item.retrieve(reader_id, {"seen": seen}, 10)}
        book = catalog[book_id]
        print(f"reader {reader_id} (read {len(seen)} books) has a cold next-read the feature tower finds:")
        print(f"  book {book_id} '{book.title}' | genres {book.fields.get('genres')} "
              f"| zero train-positive reads: {book_id in zero_train}")
        print(f"  in feature-tower top-10? {book_id in feature_recs}")
        print(f"  in ID-only two-tower top-10? {book_id in tower_recs}")
        print(f"  in item-item CF top-10? {book_id in cf_recs}")
        break
print()
print("A zero-interaction 'children' book, placed by its genre/author/keywords -- invisible to both")
print("collaborative baselines, surfaced by the feature tower from what the book IS.")

reader 129 (read 16 books) has a cold next-read the feature tower finds:
  book 139 'Book 00139' | genres children | zero train-positive reads: True
  in feature-tower top-10? True
  in ID-only two-tower top-10? False
  in item-item CF top-10? False

A zero-interaction 'children' book, placed by its genre/author/keywords -- invisible to both
collaborative baselines, surfaced by the feature tower from what the book IS.


## Where this goes

- An **id-only** two-tower (and item-item CF) buries a **cold item** -- a book with zero train-positive
  interactions -- because its id embedding was only ever shaped by being sampled as a negative.
  Measured as **cold-item coverage** (a val-safe metric, since cold items have no val positives), both
  collaborative paths surface **0** of the 818 cold books.
- The **feature tower** rebuilds the item side from **content**: `id + genre + author + Linear(GloVe)`,
  summed to the shared dimension, with the feature parameters shared across books and trained on the
  warm items -- so a cold book inherits a vector from *what it is*.
- The decisive knob is the **negative pool**. Full-catalog negatives make cold items negatives-only
  and bury them (coverage ~0.05); drawing negatives **warm-only** lifts coverage ~3x (to ~0.138) at a
  small warm cost (0.298 vs the two-tower's 0.340). **Hard** (popularity-weighted) negatives trade warm
  for cold harder still (0.172 / 0.226) and risk false negatives -- an ablation knob, not an upgrade.
- Honest headline: the feature tower is the first **learned-taste / collaborative** path to serve cold
  items while holding near the book's best warm hit -- a **compromise**, not dominance (Unit 7's
  content path already covers cold at ~0.33, but reads far worse warm).

### The bridge ahead

The feature tower finally closes the cold-start gap Unit 6 flagged -- on the retrieval side. Three
things remain. **Unit 10** makes retrieval *fast*: scoring every one of 2,000 books per reader is fine
here, but real catalogs need approximate nearest-neighbor (ANN) search. **Unit 11** adds a learned
**reranker** on top of the retrieved shortlist. And **Unit 13** returns to cold-start in the ethics /
beyond-accuracy thread -- who gets served when the catalog keeps growing, and what "coverage" owes the
books, authors and readers on the long tail.